In [3]:
from dataclasses import dataclass
from contextlib import contextmanager
import portion as P
import interval_logic as il

@dataclass
class Contract:
    assumption: il.Expression | str
    guarantee: il.Expression | str

    def __post_init__(self) -> None:
        self.assumption = il.as_expr(self.assumption)
        self.guarantee = il.as_expr(self.guarantee)

        # Maintain saturated-form invariant
        self.guarantee = il.Or((
            self.guarantee,
            il.Not(self.assumption),
        )).simplify()

    def __str__(self) -> str:
        return f"\u27E8 {self.assumption} \u2223 {self.guarantee} \u27E9"

    @staticmethod
    def merge_s(self: 'Contract', other: 'Contract') -> str:
        """Merge two contracts by combining their assumptions and guarantees."""
        return f"{self!s} \u2297 {other!s}"

    def merge(self, other: 'Contract') -> 'Contract':
        """Merge two contracts by combining their assumptions and guarantees."""
        return Contract(il.And((self.assumption, other.assumption)),
                        il.Or((il.And((self.guarantee, other.guarantee)),
                              il.Not(il.And((self.assumption, other.assumption))))))

    @staticmethod
    def separate_s(self, other: 'Contract') -> str:
        """Separate the contract into its assumption and guarantee."""
        return f"{self!s} \u29BC {other!s}"

    def separate(self, other: 'Contract') -> 'Contract':
        """Separate the contract into its assumption and guarantee."""
        return Contract(il.Or((il.And((self.assumption, other.guarantee)),
                               il.Not(il.And((self.guarantee, other.assumption))))),
                        il.And((self.guarantee, other.assumption)))

    def to_nnf(self) -> 'Contract':
        """Convert the contract to Negation Normal Form (NNF)."""
        return Contract(il.to_nnf(self.assumption), il.to_nnf(self.guarantee))
    
    def simplify(self) -> 'Contract':
        """Simplify the contract by removing redundant assumptions and guarantees."""
        return Contract(self.assumption.simplify(), self.guarantee.simplify())

    def replace(self, *fns) -> 'Contract':
        """Replace parts of the contract using a given function."""
        return Contract(self.assumption.replace(*fns), self.guarantee.replace(*fns))

    def is_consistent(self, universe: P.Interval) -> bool:
        """Check if the contract is consistent (i.e., the assumption implies the guarantee)."""
        if self.guarantee == il.Bool(True):
            return True
        return il.denote(il.And((self.assumption, self.guarantee)),
                         {}, universe) == universe

@contextmanager
def step(msg: str, disable: int = 4):

    counter = list(getattr(step, "counter", (0,)))
    counter[-1] += 1
    if len(counter) == 1:
        println(f"⏺ Step {counter[0]}: {msg}")
    else:
        println(f"⏺ Step {'.'.join(map(str, counter))}: {msg}", 
                marker=["├", "│"])
    step.counter = tuple(counter + [0])
    _disable = getattr(step, "disable", 4)
    step.disable = disable
    try:
        yield
    finally:
        pass
    step.disable = _disable
    step.counter = step.counter[:-1]
    if len(step.counter) == 1:
        println('╯', left=0)
    else:
        println('', marker=["├─╯", "│"])

def reset_step():
    step.counter = (0,)
    step.disable = 4

def println(*lines: str, **kwds):
    assert all(isinstance(line, str) for line in lines)
    lines = [""] if len(lines) == 0 else list(lines)
    marker = kwds.get("marker", "│")
    top = kwds.get("top", kwds.get("y", 0))
    bottom = kwds.get("bottom", kwds.get("y", 0))
    left = kwds.get("left", kwds.get("x", 0))
    right = kwds.get("right", kwds.get("x", 0))

    indent = len(getattr(step, "counter", (0,))) - 1
    if indent >= getattr(step, "disable", 4):
        return
    ms = [marker] if isinstance(marker, str) else list(marker)
    ms = [m + " " for m in ms]
    ms += [ms[-1]] * (indent - len(ms)) # may be more than `indent` markers
    ws = "".join(reversed(ms[:indent]))

    max_width = max(len(line) for line in lines) + left + right
    lines = [ws + (" " * left + line + " " * right).ljust(max_width)
             for line in lines]
    lines = [ws]*top + lines + [ws]*bottom

    print("\n".join(lines))

def printeq(arg, *args):
    lines = [str(arg)]
    lines += [f"= {arg!s}" for arg in args]
    println(*lines)

with step("Test"):
    println("This is a test.")
    with step("Subtest"):
        println("Hello, world!")
        printeq(1 + 2, 3, 4)
    println("Back to the main test.")
    with step("Another subtest"):
        println("Goodbye, world!")

⏺ Step 1: Test
│ This is a test.
├ ⏺ Step 1.1: Subtest
│ │ Hello, world!
│ │ 3  
│ │ = 3
│ │ = 4
├─╯ 
│ Back to the main test.
├ ⏺ Step 1.2: Another subtest
│ │ Goodbye, world!
├─╯ 
╯


In [4]:
step.counter = (0,)
UNIVERSE = P.closed(0, 5)

#??
signals = {
    'j': il.NaturalSignal({
        11: P.closed(1, 3),
    })
}

with step('Initial Contract'):
    printeq(C_cmt := Contract(il.Bool(True), il.Bool(True)))

with step('Request 1'):

    with step('Incoming Request'):
        printeq(C_req1 := Contract(il.Exists(P.from_string('[1, 2)', float), il.Signal('j').equals(11)), il.Bool(True)))
        # This type of contract is definitionally inconsistent, but 
        # with the implication-interpretation it seems correct:
        #       ⊤ -> ∃[1.0,2.0): j = 1
        #     = ∃[1.0,2.0): j = 1
        # But is it important for a request to be consistent? Perhaps not... 

    with step('Candidate Contract'):
        printeq(C_cand1 := C_cmt.merge(C_req1),
                C_cand1 := C_cand1.simplify())

    with step('Create Offer Contract'):
        if C_cand1.is_consistent(UNIVERSE):
            with step('Candidate Contract is Consistent'):
                phi = C_cand1.assumption
                if isinstance(phi, il.Exists):
                    phi = il.ForAll(phi.interval, phi.body)
                printeq(C_ofr1 := Contract(il.Bool(True), phi))

    with step('Merge Accepted Offer'):
        printeq(C_cmt := C_cmt.merge(C_ofr1),
                C_cmt := C_cmt.simplify())
    
with step('Request 2'):
        
    with step('Incoming Request'):
        printeq(C_req2 := Contract(il.Exists(P.from_string('[2, 4)', float), il.Signal('j').equals(22)), il.Bool(True)))

    with step('Candidate Contract'):
        printeq(C_cand2 := C_cmt.merge(C_req2),
                C_cand2 := C_cand2.simplify())

    

⏺ Step 1: Initial Contract
│ ⟨ ⊤ ∣ ⊤ ⟩
╯
⏺ Step 2: Request 1
├ ⏺ Step 2.1: Incoming Request
│ │ ⟨ ∃[1.0,2.0): j = 11 ∣ ⊤ ⟩
├─╯ 
├ ⏺ Step 2.2: Candidate Contract
│ │ ⟨ ⊤ ∧ (∃[1.0,2.0): j = 11) ∣ ⊤ ⟩
│ │ = ⟨ ∃[1.0,2.0): j = 11 ∣ ⊤ ⟩    
├─╯ 
├ ⏺ Step 2.3: Create Offer Contract
│ ├ ⏺ Step 2.3.1: Candidate Contract is Consistent
│ │ │ ⟨ ⊤ ∣ ∀[1.0,2.0): j = 11 ⟩
│ ├─╯ 
├─╯ 
├ ⏺ Step 2.4: Merge Accepted Offer
│ │ ⟨ ⊤ ∧ ⊤ ∣ ∀[1.0,2.0): j = 11 ⟩
│ │ = ⟨ ⊤ ∣ ∀[1.0,2.0): j = 11 ⟩  
├─╯ 
╯
⏺ Step 3: Request 2
├ ⏺ Step 3.1: Incoming Request
│ │ ⟨ ∃[2.0,4.0): j = 22 ∣ ⊤ ⟩
├─╯ 
├ ⏺ Step 3.2: Candidate Contract
│ │ ⟨ ⊤ ∧ (∃[2.0,4.0): j = 22) ∣ ¬(∃[2.0,4.0): j = 22) ∨ (∀[1.0,2.0): j = 11) ⟩
│ │ = ⟨ ∃[2.0,4.0): j = 22 ∣ ¬(∃[2.0,4.0): j = 22) ∨ (∀[1.0,2.0): j = 11) ⟩    
├─╯ 
╯


$$
\begin{split}
C^\text{cand}_1 =& \langle \top \mid \exists [1, 2): j = 1 \rangle
\end{split}
$$

In [5]:
def keep(n):
    j = il.Signal('j')
    n = il.Natural(n)
    p = il.Atom('p')
    q = il.Atom('q')
    def replacer(psi):
        match psi:
            case il.Equal(left, right):
                if left == j:
                    return  p if right == n else q
                if right == j:
                    return p if left == n else q
        return psi
    return replacer

def pos(psi):
    match psi:
        case il.And((left, right)): pass
        case _: return psi
    match (left, right):
        case (il.ForAll(I, p), il.Exists(J, q)):
            if p == il.Not(q):
                return il.Exists(J - I, q)
        case (il.Exists(J, q), il.ForAll(I, p)):
            if p == il.Not(q):
                return il.Exists(J - I, q)
    return psi

def flip(psi):
    match psi:
        case il.Exists(I, p):
            return il.ForAll(I, p)
    return psi

def postproc(phi, n):
    return phi.replace(keep(n)).replace(pos).replace(flip)


In [9]:
with step('Conflict'):

    println('Commitments')
    printeq(C_cmt := Contract(True, '∀[1,3]:j=1'))

    println('Request', top=1)
    printeq(C_req2 := Contract(True, '∀[2,4]:j=2'))

    println('Merged', top=1)
    printeq(Contract.merge_s(C_cmt, C_req2),
            C_m := C_cmt.merge(C_req2),
            C_m := C_m.replace(keep(2)),
            # C_m := C_m.replace(subst(p='j=2', q='j!=2')),
            C_m := C_m.simplify())

    println('This is false => We must repair')

    println('Weaken Request', top=1)
    def weaken(psi):
        match psi:
            case il.ForAll(I, p):
                return il.Exists(I, p)
        return psi
    printeq(f'w({C_req2!s})',
            C_req2_w := Contract(True, '∀[2,4]:j=2').replace(weaken).simplify())

    println('Separated', top=1)
    def foo(psi):
        match psi:
            case il.Or((left, right)): pass
            case _: return psi
        if isinstance(left, il.Exists) and isinstance(right, il.ForAll):
            return left if left.interval.overlaps(right.interval) else psi
        if isinstance(left, il.ForAll) and isinstance(right, il.Exists):
            return right if right.interval.overlaps(left.interval) else psi
        return psi
    def qdual(psi):
        match psi:
            case il.Not(x): pass
            case _:
                # print(psi, 'XX')
                return psi
        match x:
            case il.Exists(I, p):
                return il.ForAll(I, il.Not(p))
            case il.ForAll(I, p):
                return il.Exists(I, il.Not(p))
        # print(psi, x)
        return psi
    printeq(Contract.separate_s(C_req2_w, C_cmt),
            C_s := C_req2_w.separate(C_cmt),
            C_s := C_s.replace(keep(2)).simplify(),
            C_s := C_s.to_nnf().simplify(),
            )

    println('Merge Separated', top=1)
    printeq(Contract.merge_s(C_s, C_cmt),
            C_ms := C_s.merge(C_cmt).to_nnf().simplify())


⏺ Step 7: Conflict
│ Commitments
│ ⟨ ⊤ ∣ ∀[1.0,3.0]: j = 1 ⟩
│ 
│ Request
│ ⟨ ⊤ ∣ ∀[2.0,4.0]: j = 2 ⟩
│ 
│ Merged
│ ⟨ ⊤ ∣ ∀[1.0,3.0]: j = 1 ⟩ ⊗ ⟨ ⊤ ∣ ∀[2.0,4.0]: j = 2 ⟩  
│ = ⟨ ⊤ ∧ ⊤ ∣ (∀[1.0,3.0]: j = 1) ∧ (∀[2.0,4.0]: j = 2) ⟩
│ = ⟨ ⊤ ∧ ⊤ ∣ (∀[1.0,3.0]: q) ∧ (∀[2.0,4.0]: p) ⟩        
│ = ⟨ ⊤ ∣ (∀[1.0,3.0]: q) ∧ (∀[2.0,4.0]: p) ⟩            
│ This is false => We must repair
│ 
│ Weaken Request
│ w(⟨ ⊤ ∣ ∀[2.0,4.0]: j = 2 ⟩)
│ = ⟨ ⊤ ∣ ∃[2.0,4.0]: j = 2 ⟩ 
│ 
│ Separated
│ ⟨ ⊤ ∣ ∃[2.0,4.0]: j = 2 ⟩ ⦼ ⟨ ⊤ ∣ ∀[1.0,3.0]: j = 1 ⟩                                                                           
│ = ⟨ ⊤ ∧ (∀[1.0,3.0]: j = 1) ∨ ¬((∃[2.0,4.0]: j = 2) ∧ ⊤) ∣ ¬(¬(∃[2.0,4.0]: j = 2) ∨ (∀[1.0,3.0]: j = 1)) ∨ (∃[2.0,4.0]: j = 2) ⟩
│ = ⟨ ¬(∃[2.0,4.0]: p) ∨ (∀[1.0,3.0]: q) ∣ ¬(¬(∃[2.0,4.0]: p) ∨ (∀[1.0,3.0]: q)) ∨ (∃[2.0,4.0]: p) ⟩                              
│ = ⟨ (∀[1.0,3.0]: q) ∨ (∀[2.0,4.0]: ¬p) ∣ ¬((∀[1.0,3.0]: q) ∨ (∀[2.0,4.0]: ¬p)) ∨ (∃[2.0,4.0]: p) ⟩                              

In [20]:
with step('Mallozzi'):

    println('Commitments')
    printeq(C_cmt := Contract(True, '∀[1,3]:j=1'),
            C_cmt := C_cmt.replace(keep(2)).simplify())

    println('Request', top=1)
    printeq(C_req := Contract(True, '∀[2,4]:j=2'),
            C_req := C_req.replace(keep(2)).simplify())

    println('Weakened Request', top=1)
    printeq(C_req_w := C_req.replace(weaken).simplify())

    println('Merged', top=1)
    printeq(Contract.merge_s(C_req_w, C_cmt),
            C_m := C_req_w.merge(C_cmt).simplify())

    println('Separated', top=1)
    printeq(Contract.separate_s(C_m, C_req),
            C_s := C_m.separate(C_req).simplify(),
            C_s := C_s.to_nnf().simplify())

    println('Merged Separated', top=1)
    printeq(Contract.merge_s(C_s, C_req),
            C_ms := C_s.merge(C_req).to_nnf().simplify())

    println('Foo', top=1)
    printeq(C_foo := C_ms.replace(lambda psi: il.as_expr('¬p') if psi == il.Atom('q') else psi).simplify())

├ ⏺ Step 10.1: Mallozzi
│ │ Commitments
│ │ ⟨ ⊤ ∣ ∀[1.0,3.0]: j = 1 ⟩
│ │ = ⟨ ⊤ ∣ ∀[1.0,3.0]: q ⟩  
│ │ 
│ │ Request
│ │ ⟨ ⊤ ∣ ∀[2.0,4.0]: j = 2 ⟩
│ │ = ⟨ ⊤ ∣ ∀[2.0,4.0]: p ⟩  
│ │ 
│ │ Weakened Request
│ │ ⟨ ⊤ ∣ ∃[2.0,4.0]: p ⟩
│ │ 
│ │ Merged
│ │ ⟨ ⊤ ∣ ∃[2.0,4.0]: p ⟩ ⊗ ⟨ ⊤ ∣ ∀[1.0,3.0]: q ⟩
│ │ = ⟨ ⊤ ∣ (∀[1.0,3.0]: q) ∧ (∃[2.0,4.0]: p) ⟩  
│ │ 
│ │ Separated
│ │ ⟨ ⊤ ∣ (∀[1.0,3.0]: q) ∧ (∃[2.0,4.0]: p) ⟩ ⦼ ⟨ ⊤ ∣ ∀[2.0,4.0]: p ⟩                                                                                                                                                
│ │ = ⟨ ¬((∀[1.0,3.0]: q) ∧ (∃[2.0,4.0]: p)) ∨ (∀[2.0,4.0]: p) ∣ ¬(¬((∀[1.0,3.0]: q) ∧ (∃[2.0,4.0]: p)) ∨ (∀[2.0,4.0]: p)) ∨ (∀[1.0,3.0]: q) ∧ (∃[2.0,4.0]: p) ⟩                                                     
│ │ = ⟨ (∀[2.0,4.0]: p) ∨ (∀[2.0,4.0]: ¬p) ∨ (∃[1.0,3.0]: ¬q) ∣ ¬((∀[2.0,4.0]: p) ∨ (∀[2.0,4.0]: ¬p) ∨ (∃[1.0,3.0]: ¬q)) ∨ (∀[1.0,3.0]: q) ∧ (∃[2.0,4.0]: p) ∨ (∀[1.0,3.0]: q) ∧ (∃[2.0,4.0]: p) ∧ (∃[2.0,4.0]: 